<a href="https://colab.research.google.com/github/AIVIETNAM-AIO-HUYTRUONG/AIO-2026/blob/main/M3/ML-Base/Tree-based%20Algorithms/decision_tree_classification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Decision Tree là gì?

- Hãy tưởng tượng ta đang vẽ một sơ đồ ra quyết định. Mọi thứ bắt đầu từ **một câu hỏi duy nhất** — ví dụ, mỗi khi chuẩn bị ra khỏi nhà, ta thường phân vân có nên mang ô theo hay không. Để trả lời, ta đặt câu hỏi: "Trời có đang mưa không?". Ứng với mỗi câu trả lời "có" hoặc "không", sơ đồ sẽ **rẽ sang một nhánh khác nhau**, kèm theo một câu hỏi mới (nếu cần) — cứ như vậy cho đến khi ta chốt được quyết định cuối cùng (mang ô hay không).

- **Decision Tree (Cây quyết định)** chính là mô hình máy học mô phỏng lại quy trình "hỏi - rẽ nhánh - quyết định" đó. Tùy vào việc đầu ra (kết quả cuối cùng) là gì, ta chia Decision Tree thành hai loại:
  - Nếu đầu ra là một **nhãn rời rạc** (ví dụ: Có/Không, Đậu/Rớt), ta gọi là **Classification Tree (Cây phân loại)**.
  - Nếu đầu ra là một **giá trị số liên tục** (ví dụ: giá nhà, điểm số), ta gọi là **Regression Tree (Cây hồi quy)**.

- Trong bài này, chúng ta sẽ tập trung tìm hiểu về **Classification Tree**.

<p align="center">
  <img src="../../../Images/Decision-tree.png" alt="Decision-tree"/>
</p>

- Một điểm mạnh của Decision Tree là có thể xử lý đồng thời nhiều dạng dữ liệu đầu vào khác nhau:
  - **Dữ liệu rời rạc (Categorical data)**: Là các biến chỉ nhận một số hữu hạn giá trị, ví dụ **Đúng/Sai**, **Trời mưa/Trời nắng**. Với loại dữ liệu này, cây sẽ tạo một nhánh riêng cho từng giá trị.

  - **Dữ liệu liên tục (Continuous/Numeric data)**: Là các biến số thực, ví dụ **Tuổi**, **Điểm số**. Với loại dữ liệu này, thuật toán sẽ tự tìm ra một **ngưỡng (threshold)** phù hợp để chia dữ liệu thành hai nhóm (ví dụ: Tuổi ≤ 30 và Tuổi > 30).

  - **Dữ liệu hỗn hợp (Mixed data)**: Là trường hợp tập dữ liệu có cả biến rời rạc lẫn biến liên tục — Decision Tree có thể xử lý kết hợp cả hai loại này trong cùng một cây.

## 1. Bài toán cần giải quyết (Split Condition)

- Khi xây dựng **Decision Tree**, tại mỗi **nút (node)**, thuật toán phải trả lời câu hỏi:
  > Trong số hàng chục **đặc trưng (features)** của dữ liệu, đâu là feature tốt nhất để dùng làm điều kiện phân chia nhánh (split condition) tại nút này?

- Nói cách khác, máy tính cần một cách **so sánh và chấm điểm** các feature, rồi chọn ra feature "tốt nhất". Để làm được điều đó một cách định lượng (bằng con số cụ thể, chứ không phải cảm tính), ta cần một thước đo toán học — đó chính là **Entropy**, khái niệm nền tảng để tính ra **Information Gain** (mức độ "thông tin" mà một feature mang lại khi dùng để chia nhánh).

- Trong phần dưới đây, ta sẽ tìm hiểu kỹ về **Entropy** trước — đây là viên gạch nền tảng, cần nắm vững trước khi có thể hiểu Information Gain.

## 2. Entropy và Information Gain — Hai thước đo giúp Decision Tree chọn nhánh tốt nhất

- Khái niệm **Entropy** được Claude Shannon đưa ra năm 1948 trong lý thuyết Thông tin (Information Theory), dùng để đo mức độ **không chắc chắn (uncertainty)** hoặc **độ xáo trộn/không thuần khiết (impurity)** của một tập dữ liệu.

> *(Ở mục **2.1**, ta sẽ xây dựng thật vững khái niệm **Entropy** trước. Sau đó, ở mục **2.2**, ta sẽ dùng chính Entropy đó để định nghĩa **Information Gain** — tiêu chí giúp Decision Tree chọn ra thuộc tính tốt nhất để rẽ nhánh.)*

- Trước khi đến với Entropy, ta cần hiểu khái niệm **độ ngạc nhiên (surprise)** của một **sự kiện (event)** $E$. Xét ví dụ sau:
  - Một chiếc túi có 10 viên bi: **9 viên bi đỏ** và **1 viên bi xanh**.
  - Gọi $E_{đỏ}$ là sự kiện "rút ngẫu nhiên được bi đỏ", $E_{xanh}$ là sự kiện "rút ngẫu nhiên được bi xanh".
  - Xác suất tương ứng: $P(E_{đỏ}) = \frac{9}{10} = 0.9$ và $P(E_{xanh}) = \frac{1}{10} = 0.1$.

- Vì bi xanh hiếm hơn bi đỏ ($P(E_{xanh}) < P(E_{đỏ})$), nếu rút trúng bi xanh ta sẽ cảm thấy **bất ngờ hơn** so với rút trúng bi đỏ. Vậy làm thế nào để đo lường "độ bất ngờ" này bằng một con số cụ thể?

- **Đề xuất ban đầu**: $$\text{Surprise}(E) = \frac{1}{P(E)}$$

- Công thức này thỏa được điều kiện mong muốn đầu tiên:
  - Sự kiện càng hiếm $P(E) \downarrow$ thì độ ngạc nhiên lại càng cao $\text{Surprise}(E) \uparrow$.
  - Tuy nhiên, nó lại gặp phải hai vấn đề:
    1. **Đơn vị mơ hồ**: $P(E) = 0.1 \Rightarrow \text{Surprise} = 10$. Vậy **10** này là gì? Ta không thể nói là "10 ngạc nhiên".
    2. **Không cộng dồn được**: Với hai sự kiện độc lập $E_1, E_2$:
        - **Kỳ vọng ban đầu (mong muốn có "tính cộng dồn")**: Lượng bất ngờ khi biết cả hai biến cố cùng xảy ra phải bằng tổng lượng bất ngờ của từng biến cố riêng lẻ hợp lại.

        $$\text{Surprise}(E_1 \cap E_2) = \text{Surprise}(E_1) + \text{Surprise}(E_2)$$

        - **Nhưng thực tế**:
          - Theo xác suất, vì $E_1$ và $E_2$ độc lập nên:

            $$P(E_1 \cap E_2) = P(E_1) \times P(E_2)$$

          - Áp dụng công thức $\text{Surprise}(E) = \frac{1}{P(E)}$ vào $E_1 \cap E_2$:

          $$\text{Surprise}(E_1 \cap E_2) = \frac{1}{P(E_1 \cap E_2)} = \frac{1}{P(E_1) \times P(E_2)}$$

          - Tách phân số ra:

          $$\frac{1}{P(E_1) \times P(E_2)} = \frac{1}{P(E_1)} \times \frac{1}{P(E_2)}$$

          - Kết quả thực tế thu được là: $$\text{Surprise}(E_1 \cap E_2) = \text{Surprise}(E_1) \times \text{Surprise}(E_2)$$

        - Chúng ta mong muốn là **cộng** nhưng thực tế lại là **nhân**.

- Để giải quyết cả hai vấn đề trên, các nhà toán học (điển hình là Claude Shannon) đã chèn thêm hàm **Logarithm** vào công thức tính độ ngạc nhiên, và gọi kết quả là $I(p)$.
  - Sở dĩ Logarithm giải quyết được vấn đề "cộng dồn" là nhờ một tính chất quen thuộc: $\log_2(x \times y) = \log_2(x) + \log_2(y)$ — nghĩa là logarithm biến **phép nhân thành phép cộng**. Đây chính xác là điều ta cần.
- $I(p)$ chính là ký hiệu toán học đại diện cho Surprise(E) (độ bất ngờ / lượng thông tin) của một biến cố $E$ có xác suất xuất hiện là $p$.

- Trong **Lý thuyết thông tin (Information Theory)**:
  - $I$ là viết tắt của **Information (Thông tin)** hoặc **Self-Information (Tự thông tin)**.
  - $p$ là xác suất $P(E)$ của biến cố đó ($0 \le p \le 1$). Thay vì viết $I(P(E))$, người ta viết gọn lại thành $I(p)$.

  $$\text{Surprise}(E) = I(P(E)) = I(p) = \log_2\left(\frac{1}{P(E)}\right) = -\log_2(P(E))$$

- Nhờ tính chất trên, với hai sự kiện độc lập $E_1, E_2$, ta có được đúng tính cộng dồn mà ban đầu ta mong muốn:
$$I(E_1 \cap E_2) = -\log_2 [P(E_1)P(E_2)] = -\log_2 P(E_1) - \log_2 P(E_2) = I(E_1) + I(E_2)$$

### 2.1. Bản chất cốt lõi: Entropy $H(S)$ thực chất là gì?

- **Entropy $H(S)$** chính là **giá trị kỳ vọng (trung bình có trọng số)** của độ bất ngờ $I(p_c)$, tính trên tất cả các nhãn $c$ có trong tập dữ liệu $S$.
- **Entropy** càng lớn thì tập dữ liệu càng "không thuần" (impure), càng khó phân tách.
  > Nhắc lại: $I(p_c)$ ở đây chính là công thức $I(p) = -\log_2(p)$ đã học ở phần trên — chỉ khác là thay vì tính cho một sự kiện $E$ bất kỳ, ta áp dụng cho từng **nhãn (label) $c$**, với $p_c$ là xác suất xuất hiện của nhãn đó.

  $$H(S) = -\sum_{c \in \mathcal{C}} p_c \log_2 p_c \quad (\text{đơn vị: bit})$$

- Trong đó:
  - $S$ (Dataset): Tập dữ liệu cần đo độ hỗn loạn/độ xáo trộn.
  - $\mathcal{C}$ (Classes): Tập hợp tất cả các nhãn phân loại có thể có trong bài toán (Ví dụ: $\mathcal{C} = \{\text{Chơi}, \text{Không chơi}\}$ hoặc $\mathcal{C} = \{0, 1\}$).
  - $c$: Một nhãn phân loại cụ thể thuộc $\mathcal{C}$.
  - $p_c$: Xác suất xuất hiện của nhãn $c$ trong tập dữ liệu $S$.
  - $-\log_2 p_c$: Chính là $I(p_c)$ — lượng thông tin hay độ ngạc nhiên (Surprise) thu được khi xuất hiện nhãn $c$.
  - **Dấu $\sum$ (Tổng)**: Lấy tổng độ ngạc nhiên của từng nhãn.
  - **Đơn vị bit**: Do sử dụng hàm logarithm cơ số 2 ($\log_2$), đơn vị đo thông tin thu được tính bằng **bit**.

    

#### 2.1.1. Ví dụ minh họa bằng số cụ thể về Entropy $H(S)$

- Giả sử bạn có tập dữ liệu $S$ gồm 10 mẫu phân loại thành 2 nhãn $\{A, B\}$:

##### 2.1.1.1. Trường hợp 1: Tập dữ liệu hoàn toàn tinh khiết (Pure)

- Gồm **10** mẫu nhãn **A**: $\rightarrow p_A = \frac{10}{10} = 1$
- **0** mẫu nhãn **B**:  $\rightarrow p_B = \frac{0}{10} = 0$.

$$H(S) = -\left(1 \cdot \log_2(1) + 0 \cdot \log_2(0)\right) = -(0 + 0) = 0 \text{ bit}$$

> **Lưu ý nhỏ**: Về mặt toán học, $\log_2(0)$ thực ra không xác định (tiến tới $-\infty$). Tuy nhiên, theo quy ước trong lý thuyết thông tin, ta luôn coi $0 \cdot \log_2(0) = 0$ — vì một nhãn không bao giờ xuất hiện thì không đóng góp gì vào độ "xáo trộn" của tập dữ liệu. Quy ước này áp dụng xuyên suốt cho mọi công thức Entropy về sau.

- **Ý nghĩa**: Entropy = 0 $\rightarrow$ Tập dữ liệu sạch tuyệt đối, không có sự xáo trộn hay mơ hồ nào

##### 2.1.1.2. Trường hợp 2: Tập dữ liệu xáo trộn tối đa (Impure / Uncertain)

- Gồm **5** mẫu nhãn **A**: $\rightarrow p_A = \frac{5}{10} = 0.5$
- **5** mẫu nhãn **B**:  $\rightarrow p_B = \frac{5}{10} = 0.5$.

$$H(S) = -\left(0.5 \cdot \log_2(0.5) + 0.5 \cdot \log_2(0.5)\right) = -\left(0.5 \cdot (-1) + 0.5 \cdot (-1)\right) = 1 \text{ bit}$$

- **Ý nghĩa**: Entropy đạt giá trị cực đại (= 1 với bài toán 2 nhãn) $\rightarrow$ Tập dữ liệu vô cùng xáo trộn, dự đoán hoàn toàn ngẫu nhiên


##### 2.1.1.3. Trường hợp 3: Bài toán có 3 nhãn (ví dụ: $A, B, C$)

- Giả sử tập dữ liệu $S$ có **10 mẫu**, được phân thành 3 nhãn $A, B, C$ với số lượng lần lượt là:
  - Nhãn $A$: 5 mẫu
  - Nhãn $B$: 3 mẫu
  - Nhãn $C$: 2 mẫu

  

##### 2.1.1.4. Đồ thị của hàm Entropy:
- Đồ thị của hàm **Entropy** mô tả mối quan hệ giữa xác suất xuất hiện của nhãn ($p$) và độ xáo trộn/bất định ($H(S)$).Để dễ hình dung nhất, ta xét đồ thị **Entropy** cho bài toán nhị phân (2 nhãn $A$ và $B$), với xác suất xuất hiện nhãn $A$ là $p$, và nhãn $B$ là $1 - p$.

- **Công thức đường cong Entropy nhị phân**:

    $$H(p) = -p \log_2(p) - (1-p) \log_2(1-p)$$

  - **Trục hoành (Trục X)**: Biểu diễn xác suất $p$ (chạy từ $0.0$ đến $1.0$).
  - **Trục tung (Trục Y)**: Biểu diễn giá trị Entropy $H(p)$ tính bằng bit (chạy từ $0.0$ đến $1.0$)


<p align="center">
  <img src="../../../Images/e0cbfe2d-9702-427d-aaf9-a142635062ec.png" alt="e0cbfe2d-9702-427d-aaf9-a142635062ec"/>
</p>

- **Các điểm quan trọng trên đường cong:**
  - Tại $p = 0$ ($0\%$ mẫu nhãn $A$, $100\%$ mẫu nhãn $B$):
    - $H(p) = 0\text{ bit}$
    - Tập dữ liệu thuần khiết tuyệt đối, không có sự xáo trộn nào.
  - Tại $p = 1$ ($100\%$ mẫu nhãn $A$, $0\%$ mẫu nhãn $B$):
    - $H(p) = 0\text{ bit}$
    - Tập dữ liệu thuần khiết tuyệt đối, không có sự xáo trộn nào.
  - Tại $p = 0.5$ ($50\%$ mẫu nhãn $A$, $50\%$ mẫu nhãn $B$):
    - $H(p) = 1\text{ bit}$ (Đỉnh của đường cong)
    - Dữ liệu phân phối đều 100%, độ ngạc nhiên/không chắc chắn đạt mức tối đa.

- **Tóm lại**: Entropy $H(S)$ cho ta biết một tập dữ liệu (hay một nút bất kỳ trong cây) đang "thuần" hay đang "xáo trộn" tới mức nào — Entropy càng thấp, dữ liệu càng thuần; Entropy càng cao, dữ liệu càng lẫn lộn giữa các nhãn.


- $S$ (Dataset): Là toàn bộ tập dữ liệu gồm **10 mẫu** mà ta đang xét.
- $\mathcal{C}$ (Classes): Là tập hợp tất cả các nhãn phân loại có trong tập $S$, cụ thể ở đây $\mathcal{C} = \{A, B, C\}$
- $c$: Là một nhãn cụ thể thuộc tập $\mathcal{C}$. Trong Trường hợp 3, $c$ sẽ lần lượt nhận từng giá trị là $A$, $B$, hoặc $C$ khi lấy tổng.
- $p_c$: Là xác suất xuất hiện của nhãn $c$ trong tập dữ liệu $S$. Cụ thể:
$$p_c = \frac{\text{Số phần tử có nhãn } c}{\text{Tổng số phần tử trong tập } S}$$
  - Khi $c = A \rightarrow p_A = \frac{5}{10} = 0.5$
  - Khi $c = B \rightarrow p_B = \frac{3}{10} = 0.3$
  - Khi $c = C \rightarrow p_C = \frac{2}{10} = 0.2$

- $-\log_2 p_c$: Là **độ ngạc nhiên (Surprise) / lượng thông tin $I(p_c)$** thu được nếu rút ngẫu nhiên được một mẫu mang nhãn $c$:
  - Khi $c = A \rightarrow  I(p_A) = -\log_2(0.5) = 1 \text{ bit}$
  - Khi $c = B \rightarrow  I(p_B) = -\log_2(0.3) \approx 1.73696 \text{ bit}$
  - Khi $c = C \rightarrow  I(p_C) = -\log_2(0.2) \approx 2.32193 \text{ bit}$


- Tính Entropy $H(S)$ bằng cách khai triển đầy đủ từng phần tử
  $$H(S) = \left[ p_A \times I(p_A) \right] + \left[ p_B \times I(p_B) \right] + \left[ p_C \times I(p_C) \right]$$

- Thay các giá trị vừa tính vào:
  $$H(S) = (0.5 \times 1) + (0.3 \times 1.73696) + (0.2 \times 2.32193)$$$$H(S) = 0.5 + 0.521088 + 0.464386$$$$H(S) \approx 1.48547 \text{ bit}$$

#### 2.1.2 Một vài tính chất tổng quát của Entropy

- Từ các ví dụ trên, ta có thể rút ra một số tính chất tổng quát sau của Entropy:
-  $H(S)=0$ khi S chỉ có 1 lớp, được gọi là thuần (Pure)
-  $H(S)=1$ đạt cực đại khi $S$ có 2 nhãn phân phối đều (phân loại nhị phân).
- Với $C >2$, tức nhiều hơn 2 nhãn mà dữ liệu phân phối đều thì $H(S) = log2|C|$:
  - $\vert{}\mathcal{C}\vert{}$ (hoặc $K$): Số lượng nhãn phân loại trong tập dữ liệu (ví dụ: có 3 nhãn thì $\vert{}\mathcal{C}\vert{} = 3$, có 4 nhãn thì $\vert{}\mathcal{C}\vert{} = 4$)
  - Dữ liệu phân phối đều: Tất cả $\vert{}\mathcal{C}\vert{}$ nhãn đều có số lượng bằng nhau, tức là cơ hội xuất hiện của mỗi nhãn là như nhau.
  
  $$p_c = \frac{1}{\vert{}\mathcal{C}\vert{}} \quad \text{cho mọi nhãn } c$$

- Thay vào: $$H(S) = -\sum_{c \in \mathcal{C}} p_c \log_2 p_c$$
  - Vì tất cả $\vert{}\mathcal{C}\vert{}$ nhãn đều có $p_c = \frac{1}{\vert{}\mathcal{C}\vert{}}$, ta thay giá trị này vào công thức:

    1. Thay $p_c$ vào:
      
      $$H(S) = -\sum_{c \in \mathcal{C}} \left( \frac{1}{\vert{}\mathcal{C}\vert{}} \cdot \log_2\left(\frac{1}{\vert{}\mathcal{C}\vert{}}\right) \right)$$
    
    2. Áp dụng tính chất **Logarithm** $\log_2\left(\frac{1}{x}\right) = -\log_2(x)$:
    
      $$\log_2\left(\frac{1}{\vert{}\mathcal{C}\vert{}}\right) = -\log_2(\vert{}\mathcal{C}\vert{})$$
    
    3. Thay ngược lại vào tổng:
      
      $$H(S) = -\sum_{c \in \mathcal{C}} \left( \frac{1}{\vert{}\mathcal{C}\vert{}} \cdot \left(-\log_2(\vert{}\mathcal{C}\vert{})\right) \right)$$
      
      $$H(S) = \sum_{c \in \mathcal{C}} \left( \frac{1}{\vert{}\mathcal{C}\vert{}} \cdot \log_2(\vert{}\mathcal{C}\vert{}) \right)$$
    
    4. Lấy tổng của $\vert{}\mathcal{C}\vert{}$ phần tử giống hệt nhau: Do ta đang cộng đại lượng $\left( \frac{1}{\vert{}\mathcal{C}\vert{}} \cdot \log_2(\vert{}\mathcal{C}\vert{}) \right)$ đúng $\vert{}\mathcal{C}\vert{}$ lần:
    
      $$H(S) = \vert{}\mathcal{C}\vert{} \times \left( \frac{1}{\vert{}\mathcal{C}\vert{}} \cdot \log_2(\vert{}\mathcal{C}\vert{}) \right)$$
      
    5. Rút gọn $\vert{}\mathcal{C}\vert{}$, ta còn lại:

      $$H(S) = \log_2(\vert{}\mathcal{C}\vert{})$$

- Nếu có 2 nhãn phân phối đều ($p_1 = p_2 = 0.5$):$$H(S) = \log_2(2) = 1 \text{ bit}$$

- Nếu có 3 nhãn phân phối đều ($p_1 = p_2 = p_3 = \frac{1}{3}$):$$H(S) = \log_2(3) \approx 1.585 \text{ bit}$$

- Nếu có 4 nhãn phân phối đều ($p_1 = p_2 = p_3 = p_4 = 0.25$):$$H(S) = \log_2(4) = 2 \text{ bit}$$

  > **Đây chính là Entropy cực đại ($H_{\max}$)**

### 2.2 Information Gain (Độ tăng thông tin)
- **Information Gain (Độ tăng thông tin)** là chỉ số định lượng đo lường lượng thông tin nhận được (hay mức độ giảm độ mơ hồ/độ xáo trộn) sau khi ta phân tách tập dữ liệu bằng một thuộc tính.

#### 2.2.1. Công thức toán học

$$\text{Information Gain}(S, A) = H(S) - \sum_{v \in \text{Vals}(A)} \frac{\vert{}S_v\vert{}}{\vert{}S\vert{}} H(S_v)$$

- Trong đó:
  - $H(S)$: Entropy của tập dữ liệu ban đầu tại nút cha (đo độ xáo trộn ban đầu)
  - $A$ là thuộc tính ta định tách.
  - $\text{Vals}(A)$: Tập các giá trị có thể có của thuộc tính $A$
  - $S_v$: Tập con gồm các mẫu có giá trị $v$ của thuộc tính $A$
  - $\frac{\vert{}S_v\vert{}}{\vert{}S\vert{}}$: Trọng số (tỷ lệ mẫu) của tập con $S_v$ so với tập tổng $S$
  - $\sum \frac{\vert{}S_v\vert{}}{\vert{}S\vert{}} H(S_v)$: Entropy trung bình có trọng số của các nút con sau khi rẽ nhánh

- Nói cách khác, **Information Gain chính là mức giảm Entropy** sau khi ta tách tập dữ liệu $S$ theo thuộc tính $A$: lấy Entropy ban đầu $H(S)$ trừ đi Entropy trung bình có trọng số của các nút con. Thuộc tính nào có giá trị Information Gain càng lớn thì khả năng phân tách (giúp dữ liệu "thuần" hơn) càng tốt.

In [ ]:
import pandas as pd
try:
    from google.colab import files
    print("Đang ở Google Colab → hãy chọn file CSV để upload...")
    uploaded = files.upload()          # hiện hộp thoại chọn file
    CSV_PATH = list(uploaded.keys())[0]
    print(f"Đã upload: {CSV_PATH}")
except ImportError:
    # --- Cách 2: Chạy local / đã có file sẵn ---
    # Thay đường dẫn này bằng file CSV của bạn
    CSV_PATH = "Student_Pass_Dataset.csv"   # file mẫu đi kèm notebook
    print(f"Không phải Colab → đọc file local: {CSV_PATH}")

df = pd.read_csv(CSV_PATH)
print("\nShape:", df.shape)

Đang ở Google Colab → hãy chọn file CSV để upload...


Saving Student_Pass_Dataset.csv to Student_Pass_Dataset.csv
Đã upload: Student_Pass_Dataset.csv

Shape: (8, 4)


In [ ]:
df

,Điểm_Tốt_Nghiệp,Chứng_chỉ_Ielts,Cộng_Điểm_Dân_Tộc,Đậu_ĐH
0,12.0,Không,Không,Không
1,14.5,Không,Có,Không
2,16.0,Không,Không,Không
3,18.0,Không,Có,Không
4,20.0,Có,Không,Có
5,22.0,Không,Không,Không
6,24.0,Có,Có,Có
7,26.0,Không,Không,Có


- Tổng số mẫu $\vert{}S\vert{} = 8$
- Cột nhãn **Đậu_ĐH ($Y$)**: Có 3 mẫu "Có" (Đậu) và 5 mẫu "Không" (Trượt)

#### 2.2.2. Các bước tính toán chi tiết



##### Bước 1: Tính Entropy ban đầu tại nút cha $H(S)$
  - Công thức Entropy của tập $S$:

  $$H(S) = - p_{\text{Có}} \log_2(p_{\text{Có}}) - p_{\text{Không}} \log_2(p_{\text{Không}})$$

  - Thay số ($p_{\text{Có}} = \frac{3}{8}$, $p_{\text{Không}} = \frac{5}{8}$):

  $$H(S) = -\frac{3}{8} \log_2\left(\frac{3}{8}\right) - \frac{5}{8} \log_2\left(\frac{5}{8}\right) \approx 0.9544$$

  ---
- Con số $0.9544\text{ bit}$ chính là Entropy ban đầu của tập dữ liệu gốc $S$ (trước khi thực hiện bất kỳ phép chia hay rẽ nhánh nào), nó cho ta biết 2 thông tin quan trọng sau:

  1. Bài toán này là bài toán phân loại nhị phân (chỉ có 2 nhãn: Có và Không):
      - Mức Entropy thấp nhất là $0\text{ bit}$ (khi 100% dữ liệu thuộc về 1 nhãn).
      - Mức Entropy cao nhất có thể đạt được là $1\text{ bit}$ (khi dữ liệu chia đều 50% Có - 50% Không).
      - Giá trị $0.9544\text{ bit}$ rất sát mốc tối đa ($1\text{ bit}$), cho thấy tập dữ liệu ban đầu đang **rất hỗn xáo, mơ hồ và chưa có tính thuần khiết** (vì có 3 mẫu "Có" và 5 mẫu "Không").
  2. Trong thuật toán Cây quyết định, con số $0.9544\text{ bit}$ đóng vai trò là **mốc so sánh ban đầu**: ở các bước tiếp theo, ta sẽ tính Entropy trung bình có trọng số của các nút con sau khi tách theo từng thuộc tính, rồi lấy $0.9544$ trừ đi giá trị đó để ra **Information Gain** — thuộc tính nào giúp Entropy giảm nhiều nhất so với mốc $0.9544$ này sẽ được chọn để rẽ nhánh.

##### Bước 2: Phân tách theo thuộc tính $A = \text{Cộng điểm dân tộc}$

- Thuộc tính $A$ có tập giá trị $\text{Vals}(A) = \{\text{True (Có)}, \text{False (Không)}\}$:


- Nhánh $v = \text{True} \rightarrow S_{\text{True}}$: Gồm 3 mẫu (dòng 1, 3, 6):
  - $\vert{}S_v\vert{}$ (Tổng số mẫu trong nhánh $v$): $\vert{}S_{\text{True}}\vert{} = 3$ mẫu
  
  - $\frac{\vert{}S_{\text{True}}\vert{}}{\vert{}S\vert{}} = \frac{3}{8}$
  
  - $p_{\text{Có}}$ (Xác suất nhãn "Có" trong tập con $S_v$):
  
    $$p_{\text{Có}} = \frac{\text{Số nhãn Có}}{\vert{}S_v\vert{}} = \frac{\mathbf{1}}{\mathbf{3}}$$
  - $p_{\text{Không}}$ (Xác suất nhãn Không trong tập con $S_v$):
  
    $$p_{\text{Không}} = \frac{\text{Số nhãn Không}}{\vert{}S_v\vert{}} = \frac{\mathbf{2}}{\mathbf{3}}$$

  - Entropy nút con:
  
    \begin{aligned}
      H(S_v) &= H(S_{\text{True}}) \\
      &= -\sum_{c \in \mathcal{C}} p_c \log_2 (p_c) \\
      &= H(S) \\
      &= -p_{\text{Có}} \log_2(p_{\text{Có}}) - p_{\text{Không}} \log_2(p_{\text{Không}}) \\
      &= -\frac{1}{3} \log_2 \left(\frac{1}{3}\right) - \frac{2}{3} \log_2 \left(\frac{2}{3}\right) \\
      &\approx 0.9183
      \end{aligned}

- Nhánh $v = \text{False}$ ($S_{\text{False}}$): Gồm 5 mẫu còn lại:
  -  $\vert{}S_v\vert{}$ (Tổng số mẫu trong nhánh $v$): $\vert{}S_{\text{False}}\vert{} = 5$ mẫu

  - $\frac{\vert{}S_{\text{False}}\vert{}}{\vert{}S\vert{}} = \frac{5}{8}$

  - $p_{\text{Có}}$ (Xác suất nhãn "Có" trong tập con $S_v$):
  
    $$p_{\text{Có}} = \frac{\text{Số nhãn Có}}{\vert{}S_v\vert{}} = \frac{\mathbf{2}}{\mathbf{5}}$$

  - $p_{\text{Không}}$ (Xác suất nhãn Không trong tập con $S_v$):
  
    $$p_{\text{Không}} = \frac{\text{Số nhãn Không}}{\vert{}S_v\vert{}} = \frac{\mathbf{3}}{\mathbf{5}}$$

  - Entropy nút con:
  
  $$H(S_{\text{False}}) = -\frac{2}{5}\log_2\left(\frac{2}{5}\right) - \frac{3}{5}\log_2\left(\frac{3}{5}\right) \approx 0.9710$$

##### Bước 3: Tính Entropy trung bình có trọng số của các nút con

- Khai triển tổng $\sum$ theo các giá trị $v \in \{\text{True}, \text{False}\}$


$$\begin{aligned}
\sum_{v \in \{\text{True, False}\}} \frac{|S_v|}{|S|} H(S_v)
&= \underbrace{\frac{|S_{\text{True}}|}{|S|} \cdot H(S_{\text{True}})}_{\text{Nhánh True}}
 + \underbrace{\frac{|S_{\text{False}}|}{|S|} \cdot H(S_{\text{False}})}_{\text{Nhánh False}} \\[1.5ex]
&= \left( \frac{3}{8} \times 0.9183 \right) + \left( \frac{5}{8} \times 0.9710 \right) \\[1ex]
&= 0.3444 + 0.6069 \\[1ex]
&= 0.9513
\end{aligned}$$

##### Bước 4: Tính Information Gain $\text{IG}(S, A)$

$$\text{IG}(S, \text{Cộng điểm dân tộc}) = H(S) - \sum \frac{\vert{}S_v\vert{}}{\vert{}S\vert{}} H(S_v) = 0.9544 - 0.9513 = 0.0032$$


- Giá trị $\text{Information Gain} = 0.0032$ rất nhỏ, cho thấy thuộc tính "Cộng điểm dân tộc" làm giảm rất ít độ xáo trộn của dữ liệu và chưa phải là một thuộc tính tốt để rẽ nhánh tại nút này.

- Tiếp theo, ta xét thuộc tính thứ hai $A_2 = \text{Chứng chỉ IELTS}$ (xem chi tiết ở **Bước 2b** ngay bên dưới), rồi đến thuộc tính thứ ba $A_3 = \text{Điểm tốt nghiệp}$ ở **Bước 5**.

##### Bước 2b: Phân tách theo thuộc tính $A_2 = \text{Chứng chỉ IELTS}$

- Thuộc tính $A_2$ có tập giá trị $\text{Vals}(A_2) = \{\text{Có}, \text{Không}\}$ (học sinh có chứng chỉ IELTS hay không).

- Nhánh $v = \text{Có} \rightarrow S_{\text{Có}}$: Gồm 2 mẫu (dòng 4, 6):

  - $\vert{}S_v\vert{}$ (Tổng số mẫu trong nhánh $v$): $\vert{}S_{\text{Có}}\vert{} = 2$ mẫu

  - $\frac{\vert{}S_{\text{Có}}\vert{}}{\vert{}S\vert{}} = \frac{2}{8}$

  - $p_{\text{Có}}$ (Xác suất nhãn "Có" — tức Đậu ĐH — trong tập con $S_v$):

    $$p_{\text{Có}} = \frac{\text{Số nhãn Có}}{\vert{}S_v\vert{}} = \frac{\mathbf{2}}{\mathbf{2}} = 1$$

  - $p_{\text{Không}}$ (Xác suất nhãn "Không" trong tập con $S_v$):

    $$p_{\text{Không}} = \frac{\text{Số nhãn Không}}{\vert{}S_v\vert{}} = \frac{\mathbf{0}}{\mathbf{2}} = 0$$

  - Entropy nút con: cả 2 học sinh có IELTS đều **Đậu ĐH**, nên nhánh này hoàn toàn thuần khiết:

    $$H(S_{\text{Có}}) = -1 \log_2(1) - 0 \log_2(0) = 0$$

- Nhánh $v = \text{Không} \rightarrow S_{\text{Không}}$: Gồm 6 mẫu còn lại (dòng 0, 1, 2, 3, 5, 7):

  - $\vert{}S_v\vert{}$ (Tổng số mẫu trong nhánh $v$): $\vert{}S_{\text{Không}}\vert{} = 6$ mẫu

  - $\frac{\vert{}S_{\text{Không}}\vert{}}{\vert{}S\vert{}} = \frac{6}{8}$

  - $p_{\text{Có}}$:

    $$p_{\text{Có}} = \frac{\text{Số nhãn Có}}{\vert{}S_v\vert{}} = \frac{\mathbf{1}}{\mathbf{6}}$$

  - $p_{\text{Không}}$:

    $$p_{\text{Không}} = \frac{\text{Số nhãn Không}}{\vert{}S_v\vert{}} = \frac{\mathbf{5}}{\mathbf{6}}$$

  - Entropy nút con:

    $$H(S_{\text{Không}}) = -\frac{1}{6}\log_2\left(\frac{1}{6}\right) - \frac{5}{6}\log_2\left(\frac{5}{6}\right) \approx 0.6500$$

- **Entropy trung bình có trọng số** của 2 nhánh:

  $$\sum \frac{\vert{}S_v\vert{}}{\vert{}S\vert{}} H(S_v) = \left(\frac{2}{8} \times 0\right) + \left(\frac{6}{8} \times 0.6500\right) = 0.4875$$

- **Information Gain**:

  $$\text{IG}(S, \text{Chứng chỉ IELTS}) = H(S) - 0.4875 = 0.9544 - 0.4875 = 0.4669$$

- So với thuộc tính "Cộng điểm dân tộc" ($\text{IG} = 0.0032$), Information Gain của "Chứng chỉ IELTS" ($0.4669$) cao hơn rất nhiều — cho thấy đây là một thuộc tính phân tách tốt: chỉ riêng việc có/không có chứng chỉ IELTS cũng đủ để tách hẳn 2 học sinh chắc chắn đậu ra khỏi phần còn lại của tập dữ liệu.

##### Bước 5: Tính Information Gain cho thuộc tính $A_3 = \text{Điểm tốt nghiệp}$ (Thuộc tính số liên tục)

- Với thuộc tính liên tục, thuật toán Decision Tree sẽ tìm một **ngưỡng chia (threshold $\theta$)** tối ưu nhất


###### Bước 5.1: Tìm ngưỡng chia $\theta$

- Sắp xếp giá trị điểm tốt nghiệp tăng dần và xét các điểm trung bình giữa các giá trị liền kề:
- Các ngưỡng có thể xét: $\theta \in \{13.25, 15.25, 17.0, 19.0, 21.0, 23.0, 25.0\}$
- Về nguyên tắc, thuật toán sẽ tính Information Gain cho **cả 7 ngưỡng** này rồi chọn ra ngưỡng cho IG lớn nhất. Để không làm dài bài, ở đây ta chỉ trình bày chi tiết cách tính với ngưỡng $\theta = 19.0$ — đây cũng chính là ngưỡng cho kết quả tốt nhất trong số 7 ngưỡng trên (các ngưỡng còn lại được tính hoàn toàn tương tự).

----

- **Trường hợp thuộc tính $A = \text{Điểm tốt nghiệp}$ với ngưỡng $\theta = 19.0$**:
  - Thuộc tính $A$ có tập giá trị $\text{Vals}(A) = \{\text{True (Điểm } > 19.0\text{)}, \text{False (Điểm } \le 19.0\text{)}\}$
  - Nhánh $v = \text{True} \rightarrow S_{\text{True}}$ (Các mẫu có điểm tốt nghiệp $> 19.0$): Gồm 4 mẫu (4, 5, 6, 7):

    - $\vert{}S_v\vert{}$ (Tổng số mẫu trong nhánh $v$): $\vert{}S_{\text{True}}\vert{} = 4$ mẫu

    - $\frac{\vert{}S_{\text{True}}\vert{}}{\vert{}S\vert{}} = \frac{4}{8}$

    - $p_{\text{Có}}$ (Xác suất nhãn "Có" trong tập con $S_v$):
      
      $$p_{\text{Có}} = \frac{\text{Số nhãn Có}}{\vert{}S_v\vert{}} = \frac{3}{4}$$

    - $p_{\text{Không}}$ (Xác suất nhãn "Không" trong tập con $S_v$):
    
      $$p_{\text{Không}} = \frac{\text{Số nhãn Không}}{\vert{}S_v\vert{}} = \frac{1}{4}$$

    - Entropy nút con:

    \begin{aligned}
      H(S_v) &= H(S_{\text{True}}) \\[1.2ex]
      &= -\sum_{c \in \mathcal{C}} p_c \log_2 (p_c) \\[1.2ex]
      &= -p_{\text{Có}} \log_2(p_{\text{Có}}) - p_{\text{Không}} \log_2(p_{\text{Không}}) \\[1.2ex]
      &= -\frac{3}{4} \log_2 \left(\frac{3}{4}\right) - \frac{1}{4} \log_2 \left(\frac{1}{4}\right) \\[1.2ex]
      &\approx 0.8113
      \end{aligned}
  - Nhánh $v = \text{False} \rightarrow S_{\text{False}}$ (Các mẫu có điểm tốt nghiệp $\le 19.0$): Gồm 4 mẫu còn lại:

    - $\vert{}S_v\vert{}$ (Tổng số mẫu trong nhánh $v$): $\vert{}S_{\text{False}}\vert{} = 4$ mẫu

    - $\frac{\vert{}S_{\text{False}}\vert{}}{\vert{}S\vert{}} = \frac{4}{8}$

    - $p_{\text{Có}}$ (Xác suất nhãn "Có" trong tập con $S_v$):
    
      $$p_{\text{Có}} = \frac{\text{Số nhãn Có}}{\vert{}S_v\vert{}} = \frac{0}{4} = 0$$

    - $p_{\text{Không}}$ (Xác suất nhãn "Không" trong tập con $S_v$):
      
      $$p_{\text{Không}} = \frac{\text{Số nhãn Không}}{\vert{}S_v\vert{}} = \frac{4}{4} = 1$$
    
    - Entropy nút con:

    \begin{aligned}
      H(S_v) &= H(S_{\text{False}}) \\[1.2ex]
      &= -\sum_{c \in \mathcal{C}} p_c \log_2 (p_c) \\[1.2ex]
      &= -p_{\text{Có}} \log_2(p_{\text{Có}}) - p_{\text{Không}} \log_2(p_{\text{Không}}) \\[1.2ex]
      &= -0 \log_2(0) - 1 \log_2(1) \\[1.2ex]
      &= 0
      \end{aligned}

###### Bước 5.2: Tính Entropy trung bình có trọng số & IG với ngưỡng $\theta = 19.0$

- Entropy trung bình có trọng số:

$$\sum \frac{\vert{}S_v\vert{}}{\vert{}S\vert{}} H(S_v) = \left(\frac{4}{8} \times 0\right) + \left(\frac{4}{8} \times 0.8113\right) = 0.4057$$

- Information Gain:

$$\text{IG}(S, \text{Điểm tốt nghiệp} \le 19.0) = H(S) - 0.4057 = 0.9544 - 0.4057 = 0.5487$$

#### 2.2.3. Bảng so sánh tổng hợp và Kết luận chọn nút

| Thuộc tính ($A$) | Thuộc tính con / Điều kiện | Entropy trung bình các nút con | Information Gain ($\text{IG}$) |
| :--- | :--- | :---: | :---: |
| **Cộng điểm dân tộc** | Có / Không | $0.9513$ | **$0.0032$** |
| **Chứng chỉ IELTS** | Có / Không | $0.4875$ | **$0.4669$** |
| **Điểm tốt nghiệp** | Điểm $\le 19.0$ | $0.4057$ | **$0.5487$** |

##### **Kết luận quyết định rẽ nhánh tại nút gốc (Root Node):**

Thuật toán **Decision Tree** sẽ chọn thuộc tính **Điểm tốt nghiệp (với ngưỡng $\le 19.0$)** làm điều kiện phân tách đầu tiên tại nút gốc vì đạt giá trị **$\text{Information Gain} = 0.5487$ lớn nhất**.

Điều này thể hiện rằng việc phân chia dữ liệu dựa trên thuộc tính *Điểm tốt nghiệp* giúp giảm độ xáo trộn (Entropy) của tập dữ liệu hiệu quả nhất so với các thuộc tính còn lại.

## 3. Gini Impurity & Gini Gain

### 3.1. Gini Impurity

- **Gini Impurity** đo lường xác suất một mẫu dữ liệu chọn ngẫu nhiên bị phân loại sai nhãn nếu ta gắn nhãn ngẫu nhiên cho mẫu đó theo phân phối xác suất của tập dữ liệu

  - **Ý nghĩa thực tế**: Giống như Entropy, Gini Impurity dùng để đo độ xáo trộn/độ không thuần khiết của dữ liệu. Giá trị Gini càng nhỏ, tập dữ liệu càng thuần khiết.

  - **Đặc điểm so với Entropy**: Gini Impurity không tính phép toán logarit ($\log_2$) nên máy tính xử lý nhanh hơn nhiều

- **Công thức toán học**:

  $$\text{Gini}(S) = 1 - \sum_{c \in \mathcal{C}} p_c^2$$

  - Trong đó:
    - $S$: Tập dữ liệu đang xét
    - $\mathcal{C}$: Tập các nhãn mục tiêu (ví dụ: $\mathcal{C} = \{\text{"Có"}, \text{"Không"}\}$).
    - $p_c$: Xác suất xuất hiện của nhãn $c$ trong tập $S$.

### 3.2. Gini Gain

- **Gini Gain** (hoặc mức giảm độ tạp chất Gini) đo lường mức độ giảm độ xáo trộn của dữ liệu sau khi ta rẽ nhánh theo một thuộc tính $A$

- **Công thức toán học**:

  $$\text{Gini Gain}(S, A) = \text{Gini}(S) - \sum_{v \in \text{Vals}(A)} \frac{\vert{}S_v\vert{}}{\vert{}S\vert{}} \text{Gini}(S_v)$$

  - Trong đó:
    - $\text{Gini}(S)$: Độ tạp chất Gini của nút cha ban đầu.

    - $\frac{\vert{}S_v\vert{}}{\vert{}S\vert{}}$: Trọng số (tỷ lệ mẫu) của nhánh con $v$

    - $\text{Gini}(S_v)$: Độ tạp chất Gini của nhánh con $v$.

    ----
    - Thuật toán sẽ ưu tiên chọn thuộc tính có Gini Gain lớn nhất để làm nút rẽ nhánh

### 3.3 Ví dụ tính toán từng bước chi tiết với Gini

- Nhìn lại tập dữ liệu $S$ gồm 8 mẫu ($\vert{}S\vert{} = 8$) với 3 mẫu "Có" và 5 mẫu "Không"


#### Bước 1: Tính Gini ban đầu tại nút cha $\text{Gini}(S)$
- $p_{\text{Có}} = \frac{3}{8} = 0.375$

- $p_{\text{Không}} = \frac{5}{8} = 0.625$

\begin{aligned}
\text{Gini}(S) &= 1 - \left( p_{\text{Có}}^2 + p_{\text{Không}}^2 \right) \\[1.2ex]
&= 1 - \left( 0.375^2 + 0.625^2 \right) \\[1.2ex]
&= 1 - (0.140625 + 0.390625) \\[1.2ex]
&= 1 - 0.53125 \\[1.2ex]
&= \mathbf{0.46875}
\end{aligned}

---
- (Lưu ý: Với phân loại nhị phân, Gini tối đa đạt được khi chia đều 50-50 là $1 - (0.5^2 + 0.5^2) = 0.5$).

#### Bước 2: Tính Gini cho Thuộc tính $A = \text{Điểm tốt nghiệp}$ (với ngưỡng $\le 19.0$)
- Thuộc tính chia thành 2 nhánh:

  - Nhánh $v = \text{True}$ ($S_{> 19.0}$): Gồm 4 mẫu (dòng 4, 5, 6, 7) $\rightarrow$ 3 mẫu "Có", 1 mẫu "Không".
  
    - $\vert{}S_{\text{True}}\vert{} = 4$
    
    - $p_{\text{Có}} = \frac{3}{4} = 0.75$

    - $p_{\text{Không}} = \frac{1}{4} = 0.25$

    $$\text{Gini}(S_{\text{True}}) = 1 - \left( 0.75^2 + 0.25^2 \right) = 1 - (0.5625 + 0.0625) = \mathbf{0.375}$$
  
  - Nhánh $v = \text{False}$ ($S_{\le 19.0}$): Gồm 4 mẫu (dòng 1, 2, 3, 4) $\rightarrow$ 0 mẫu "Có", 4 mẫu "Không".

    - $\vert{}S_{\text{False}}\vert{} = 4$

    - $p_{\text{Có}} = \frac{0}{4} = 0$

    - $p_{\text{Không}} = \frac{4}{4} = 1$

      $$\text{Gini}(S_{\text{False}}) = 1 - \left( 0^2 + 1^2 \right) = 1 - 1 = \mathbf{0}$$

#### Bước 3: Tính Gini trung bình có trọng số của các nút con

$$\sum \frac{\vert{}S_v\vert{}}{\vert{}S\vert{}} \text{Gini}(S_v) = \left( \frac{4}{8} \times 0.375 \right) + \left( \frac{4}{8} \times 0 \right) = (0.5 \times 0.375) + 0 = \mathbf{0.1875}$$

#### Bước 4: Tính Gini Gain

$$\text{Gini Gain}(S, \text{Điểm tốt nghiệp} \le 19.0) = \text{Gini}(S) - \sum \frac{\vert{}S_v\vert{}}{\vert{}S\vert{}} \text{Gini}(S_v)  = 0.46875 - 0.1875 = \mathbf{0.28125}
$$